# Stage 25A: temporal path decoder screen

Stage 24Aの5 checkpointsを固定し、offsetを各row独立に使わず連続pathとして復号します。62 design-validation cutsだけを使うexploratory screenで、予約120 wellsは触りません。eligible profileがあっても提出や予約well確認へ直接進まず、500-cut OOF nested auditを先に行います。T4/L4で十分です。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess,sys
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
import torch
assert torch.cuda.is_available(),'Select a GPU runtime and reconnect'
assert (data_dir/'train').is_dir(),data_dir
print('GPU:',torch.cuda.get_device_name(0),'PyTorch:',torch.__version__)


## Stage 24A checkpoints

TCNは再学習せず、5 checkpointsをそのまま使います。


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage21b_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
stage24a_run=artifact_dir/'stage24a_scaled_ordinal_emission_full_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',public_oof_run/'base_oof.parquet',stage21b_run/'confidence_cut_report.parquet',stage24a_run/'summary.json']+[stage24a_run/f'fold_{fold}.pt' for fold in range(5)]
for path in required: assert path.is_file(),path
stage24_summary=json.loads((stage24a_run/'summary.json').read_text())
assert stage24_summary['stage24a_complete'] and not stage24_summary['promoted_to_stage24b_reserved_confirmation']
print(*required,sep='\n')


## Fixed path profiles

posterior mean平滑、posterior median、3つのViterbi profileを事前固定比較します。


In [ ]:
RUN_ID='stage25a_temporal_path_decoder_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve()
    assert resolved==expected and resolved.parent==artifact_dir.resolve(),resolved
    print('Removing incomplete prior run:',resolved); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    command=[sys.executable,'-m','rogii.cli.emission_path_decoder','--config','configs/experiment/stage25a_temporal_path_decoder.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--stage24a-run',str(stage24a_run),'--validation-run',str(stage21b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID,'--device','cuda']
    decoder_env=os.environ.copy()
    decoder_env['PYTHONPATH']=str(repo_dir/'src')+':'+decoder_env.get('PYTHONPATH','')
    log_path=artifact_dir/f'{RUN_ID}_driver.log'
    with log_path.open('w',encoding='utf-8') as log_handle:
        process=subprocess.Popen(command,cwd=repo_dir,env=decoder_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end=''); log_handle.write(line); log_handle.flush()
        return_code=process.wait()
    if return_code!=0:
        print('\n'.join(log_path.read_text(encoding='utf-8',errors='replace').splitlines()[-200:]))
        raise RuntimeError(f'Stage 25A failed with exit code {return_code}. Full log: {log_path}')
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage25a_complete','promoted_to_stage25b_oof_audit','device','design_validation_cuts','design_validation_wells','selected_screen_profile','screen_role','reserved_confirmation_used','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['profiles']).sort_values(['eligible','rmse'],ascending=[False,True])


summary辞書とprofile表を共有してください。この画面の最良profileをそのまま採用せず、eligibleの場合も500-cut OOF nested auditへ進みます。
